# 02. Text Normalization, Tokenization & Lemmatization
**Context-Aware Movie Review Sentiment Analyzer**  
*Role: NLP Data Preprocessing Team Member*

---

### Objectives
1. **Explain the Preprocessing Philosophy**: Why standard NLP pipelines fail on sentiment analysis (destroying negations and clause boundaries).
2. **Modular Text Cleaning**: HTML tag stripping, URL removal, and whitespace normalization.
3. **Contraction Expansion**: Revealing implicit negation tokens (e.g., `wasn't` -> `was not`).
4. **Context-Preserving Tokenization**: Keeping sentence and clause delimiters (`.`, `!`, `?`, `,`, `;`) for downstream negation scope tracking.
5. **Negation Preservation in Stopwords**: Demonstrating the critical difference between naive stopword removal and negation-preserving filtering.
6. **Part-of-Speech-Guided Lemmatization**: Reducing words to base forms using WordNet with Treebank-to-WordNet POS mapping.
7. **End-to-End Pipeline (`preprocess_text`)**: Validating the full pipeline on real IMDb sample reviews.

In [1]:
import os
import sys
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing import (
    clean_text,
    tokenize_text,
    lemmatize_tokens,
    remove_stopwords,
    preprocess_text,
    expand_contractions,
    NEGATION_WORDS,
    CONTRACTION_MAP,
)

print("Preprocessing module imported successfully.")

Preprocessing module imported successfully.


## 1. Modular Cleaning & Contraction Expansion
Standard text cleaning often strips apostrophes, turning `didn't` into `didnt` or `did nt`, which causes stopword filters to drop `did` and leave `nt` as out-of-vocabulary noise.
Expanding contractions beforehand explicitly recovers the canonical negation token `not`.

In [2]:
sample_raw = "I didn't like the ending!<br /><br />It wasn’t worth the hype, and you won't either."

print("Original Raw Review:")
print(sample_raw)

expanded = expand_contractions(sample_raw)
print("\nAfter Contraction Expansion:")
print(expanded)

cleaned = clean_text(sample_raw)
print("\nAfter clean_text (HTML stripped, normalized punctuation, lowercased):")
print(cleaned)

Original Raw Review:
I didn't like the ending!<br /><br />It wasn’t worth the hype, and you won't either.

After Contraction Expansion:
I did not like the ending!<br /><br />It was not worth the hype, and you will not either.

After clean_text (HTML stripped, normalized punctuation, lowercased):
i did not like the ending ! it was not worth the hype , and you will not either .


## 2. Tokenization with Boundary Preservation
Why preserve punctuation marks like `.`, `!`, `?`, `,` as discrete tokens?
- **Negation Scope Delimitation**: In context-aware NLP (module 03), a negation word like `not` only negates words up to the next clause boundary (comma, semicolon, period). If punctuation is erased, negation spills over into subsequent unrelated clauses.
- **Sentiment Intensity**: Exclamation marks (`!`) signal strong subjective sentiment; question marks (`?`) frequently indicate rhetorical doubt or sarcasm.

In [3]:
tokens_with_punct = tokenize_text(cleaned, preserve_punctuation=True)
tokens_no_punct = tokenize_text(cleaned, preserve_punctuation=False)

print("Tokens with Punctuation Preserved:")
print(tokens_with_punct)

print("\nTokens with Punctuation Stripped:")
print(tokens_no_punct)

Tokens with Punctuation Preserved:
['i', 'did', 'not', 'like', 'the', 'ending', '!', 'it', 'was', 'not', 'worth', 'the', 'hype', ',', 'and', 'you', 'will', 'not', 'either', '.']

Tokens with Punctuation Stripped:
['i', 'did', 'not', 'like', 'the', 'ending', 'it', 'was', 'not', 'worth', 'the', 'hype', 'and', 'you', 'will', 'not', 'either']


## 3. Stopword Filtering & Negation Preservation
NLTK's standard English stopword list contains 198 words, including:
`'not'`, `'no'`, `'nor'`, `'neither'`, `'wasn't'`, `'didn't'`, `'wouldn't'`, etc.
If we naively remove all NLTK stopwords, the sentiment of negative reviews gets completely inverted!

In [4]:
# Inspect negation words in NLTK's default stopword set
nltk_sw = set(stopwords.words('english'))
present_negations = nltk_sw.intersection(NEGATION_WORDS)
print(f"Negation words present in standard NLTK stopwords: {sorted(list(present_negations))}")

# Sentiment Inversion Demonstration
inverted_example = "The movie was not good and I did not enjoy it."
tokens_example = tokenize_text(clean_text(inverted_example))

# Naive stopword removal (removing all NLTK stopwords)
naive_filtered = [t for t in tokens_example if t not in nltk_sw]

# Context-aware stopword removal (preserving negations)
smart_filtered = remove_stopwords(tokens_example, preserve_negations=True)

print(f"\nRaw Review:               '{inverted_example}'")
print(f"Naive Stopword Removal:   {naive_filtered}  <-- CRITICAL ERROR: Sentiment inverted to 'good', 'enjoy'!")
print(f"Negation-Aware Filtering: {smart_filtered}  <-- ACCURATE: 'not' preserved before sentiment words!")

Negation words present in standard NLTK stopwords: ['no', 'nor', 'not']

Raw Review:               'The movie was not good and I did not enjoy it.'
Naive Stopword Removal:   ['movie', 'good', 'enjoy', '.']  <-- CRITICAL ERROR: Sentiment inverted to 'good', 'enjoy'!
Negation-Aware Filtering: ['movie', 'not', 'good', 'not', 'enjoy', '.']  <-- ACCURATE: 'not' preserved before sentiment words!


## 4. Lemmatization: POS-Guided vs. Simple
Lemmatization reduces words to their grammatical lemma. Using Part-of-Speech (POS) tagging ensures:
- Verbs: `was` -> `be`, `running` -> `run`, `watched` -> `watch`
- Adjectives: `better` -> `good`
- Nouns: `movies` -> `movie`, `actors` -> `actor`
- Punctuation marks are kept intact.

In [5]:
test_tokens = ["the", "actors", "were", "running", "better", "in", "recent", "movies", "!"]

lemmas_pos = lemmatize_tokens(test_tokens, use_pos=True)
lemmas_simple = lemmatize_tokens(test_tokens, use_pos=False)

comparison_df = pd.DataFrame({
    'Token': test_tokens,
    'Simple Lemma (Noun-only)': lemmas_simple,
    'POS-Guided Lemma': lemmas_pos
})
comparison_df

,Token,Simple Lemma (Noun-only),POS-Guided Lemma
0,the,the,the
1,actors,actor,actor
2,were,were,be
3,running,running,run
4,better,better,well
5,in,in,in
6,recent,recent,recent
7,movies,movie,movie
8,!,!,!


## 5. End-to-End Preprocessing (`preprocess_text`)
We test `preprocess_text()` on three classic sentiment review archetypes:
1. Direct Positive
2. Heavily Negated Negative
3. Contrastive Review (incorporating `but`)

In [6]:
reviews = [
    "A truly masterpiece! Stunning visuals, brilliant acting, and deeply emotional storytelling.",
    "This was not good at all. In fact, it wasn't worth the ticket price, and I wouldn't recommend it to anyone!",
    "The cinematography was gorgeous, but the plot was not interesting and the characters were poorly written."
]

for i, rev in enumerate(reviews, 1):
    processed_str = preprocess_text(rev, return_tokens=False)
    processed_tokens = preprocess_text(rev, return_tokens=True)
    print(f"--- Example {i} ---")
    print(f"Raw:         {rev}")
    print(f"Processed:   {processed_str}")
    print(f"Tokens:      {processed_tokens}\n")

--- Example 1 ---
Raw:         A truly masterpiece! Stunning visuals, brilliant acting, and deeply emotional storytelling.
Processed:   truly masterpiece ! stunning visuals , brilliant acting , deeply emotional storytelling .
Tokens:      ['truly', 'masterpiece', '!', 'stunning', 'visuals', ',', 'brilliant', 'acting', ',', 'deeply', 'emotional', 'storytelling', '.']

--- Example 2 ---
Raw:         This was not good at all. In fact, it wasn't worth the ticket price, and I wouldn't recommend it to anyone!
Processed:   not good . fact , not worth ticket price , would not recommend anyone !
Tokens:      ['not', 'good', '.', 'fact', ',', 'not', 'worth', 'ticket', 'price', ',', 'would', 'not', 'recommend', 'anyone', '!']

--- Example 3 ---
Raw:         The cinematography was gorgeous, but the plot was not interesting and the characters were poorly written.
Processed:   cinematography gorgeous , plot not interesting character poorly write .
Tokens:      ['cinematography', 'gorgeous', ',', 'pl

## 6. Real Dataset Batch Preprocessing Demonstration
We load the pre-split training data `data/processed/train.csv` and apply `preprocess_text` to sample reviews.

In [7]:
train_path = "../data/processed/train.csv"
train_df = pd.read_csv(train_path, nrows=500)

print(f"Loaded {len(train_df)} sample training reviews.")

# Process reviews
train_df['cleaned_review'] = train_df['review'].apply(lambda x: preprocess_text(x, return_tokens=False))

# Display side-by-side comparison
for idx in range(3):
    print(f"\n--- Sample {idx+1} ({train_df['sentiment'].iloc[idx].upper()}) ---")
    print(f"Original:    {train_df['review'].iloc[idx][:200]}...")
    print(f"Preprocessed: {train_df['cleaned_review'].iloc[idx][:200]}...")

Loaded 500 sample training reviews.



--- Sample 1 (POSITIVE) ---
Original:    I am one of Jehovah's Witnesses and I also work in an acute care medical facility. Over the years I have seen people die from hemolytic reactions to blood transfusions, have attended numerous conferen...
Preprocessed: one jehovah 's witness also work acute care medical facility . year see people die hemolytic reaction blood transfusion , attend numerous conference blood bear pathogen , see several patient become se...

--- Sample 2 (POSITIVE) ---
Original:    Amongst the standard one liner type action films, where acting and logic are checked at the door, this movie is at the top of the class. If the person in charge of casting were to have put "good" acto...
Preprocessed: amongst standard one liner type action film , act logic check door , movie top class . person charge cast put good actor flick , would worse except richard dawson actually act well , call play act . l...

--- Sample 3 (NEGATIVE) ---
Original:    Realty television crew are ass

## 7. Collaboration Contract & Export Verification
The preprocessing module is ready for team members owning:
- **Module 03: Context-Aware Feature Engineering** (`src/context_features.py`)
- **Module 04: TF-IDF Vectorization** (`src/vectorizer.py`)
- **Module 05: Model Training** (`src/train.py`)

### Exported Function Reference
| Function | Input | Output | Purpose |
| :--- | :--- | :--- | :--- |
| `clean_text(text)` | `str` | `str` | HTML stripping, contraction expansion, whitespace normalization |
| `tokenize_text(text)` | `str` | `list[str]` | Word and punctuation tokenization preserving clause boundaries |
| `remove_stopwords(tokens)` | `list[str]` | `list[str]` | Stopword removal strictly preserving negation tokens |
| `lemmatize_tokens(tokens)` | `list[str]` | `list[str]` | POS-guided WordNet lemmatization |
| `preprocess_text(text)` | `str` | `str` or `list[str]` | End-to-end normalization pipeline |
| `split_data(df)` | `pd.DataFrame` | `(train_df, test_df)` | Deterministic stratified 80:20 train-test split |